# VieNeu-TTS API · Colab One Click + Cloudflare

1. Chọn **Runtime → Change runtime type → GPU**.
2. Upload **toàn bộ checkpoint đã merge** lên Google Drive. Nhập `BACKBONE_PATH` trong form bên dưới.
3. Bấm **▶ Chạy API + Cloudflare Tunnel**, cấp quyền Drive và nhập API key ẩn (hoặc tạo Colab Secret `VIENEU_API_KEY`).
4. Khi thấy **SẴN SÀNG**, copy API URL vào Voice-Pro. Dùng `POST /v1/tts`; OpenAI dùng `/v1` và `stream:false`.

Chỉ hỗ trợ checkpoint **v3 Turbo PyTorch**; adapter LoRA cần merge trước. Ví dụ:

```text
/content/drive/MyDrive/VieNeu/dat_phi_01/merged/
├── voices_v3_turbo.json
└── update/
    ├── model.safetensors
    ├── config.json
    ├── tokenizer.json
    └── tokenizer_config.json
```

Có thể nhập đường dẫn `merged` hoặc `merged/update`; file giọng phải nằm ở `merged`. Model được đọc từ Drive; runtime còn tải codec/speaker encoder từ Hugging Face lần đầu.

**Source API:** mặc định clone `REPO_URL` / `REPO_REF`. Repo đó phải chứa tính năng local checkpoint (commit `e8960c4` hoặc thay đổi tương đương). Nếu code mới chưa push lên GitHub, upload source dự án lên Drive và nhập `SOURCE_DIR` (thư mục chứa `apps/`, `src/`, `pyproject.toml`, `uv.lock`). Không upload `.venv` và `.git`. Notebook tự tạo config riêng, không sửa `config.yaml` trên Drive.

**Phiên chạy:** URL Quick Tunnel đổi mỗi lần mở và ngừng khi Colab/tunnel dừng; không có bảo đảm uptime, không hỗ trợ SSE. Request audio dài có thể gặp timeout của proxy, nên chia văn bản thành request ngắn. Đây là notebook dùng trong phiên tương tác; Colab có thể ngắt runtime, đặc biệt khi dùng dịch vụ bên ngoài notebook trên gói miễn phí. Không có cơ chế chống idle.

Nguồn: [Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Sau khi chạy server, dùng cell **Lấy danh sách voices từ API** để lấy ID chính xác. Cell audio mẫu có `TEMPERATURE` tùy chọn (0–2, mặc định 0.8). API audio file tự cân peak về tối đa 0.98 trước encode như chức năng cân biên độ của Web; audio nhỏ giữ nguyên mức. Source server phải có thay đổi này để áp dụng.


In [ ]:
# @title ▶ Chạy API + Cloudflare Tunnel { display-mode: "form" }
BACKBONE_PATH = "/content/drive/MyDrive/VieNeu/dat_phi_01/merged" # @param {type:"string"}
# SOURCE_DIR: thư mục source đã upload lên Drive; để trống để clone GitHub.
SOURCE_DIR = "" # @param {type:"string"}
REPO_URL = "https://github.com/pnnbao97/VieNeu-TTS.git" # @param {type:"string"}
REPO_REF = "dev" # @param {type:"string"}
# Nên lấy API key từ Colab Secrets (tên VIENEU_API_KEY), hoặc nhập ẩn khi được hỏi.
API_KEY_SECRET = "VIENEU_API_KEY" # @param {type:"string"}
STARTUP_TIMEOUT_SECONDS = 1200 # @param {type:"integer"}

import getpass
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from google.colab import drive, userdata

# Dừng các process của lần chạy trước, kể cả khi lần trước chưa hoàn tất.
def stop_server():
    for name in ("tunnel_process", "server_process"):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    for name in ("server_log_handle", "tunnel_log_handle"):
        handle = globals().get(name)
        if handle is not None:
            handle.close()

stop_server()
PUBLIC_URL = None
BASE_URL = None
CAPABILITIES = None
print("1/6 Mount Google Drive (hãy cấp quyền trong hộp thoại)...")
drive.mount("/content/drive")
checkpoint = Path(BACKBONE_PATH.strip()).expanduser().resolve()
if not checkpoint.is_dir():
    raise ValueError(f"Không tìm thấy thư mục backbone: {checkpoint}")
# Chấp nhận merged/ hoặc merged/update/. Giữ toàn bộ checkpoint trên Drive.
if (checkpoint / "update/model.safetensors").is_file():
    root, subfolder = checkpoint, "update"
elif (checkpoint / "model.safetensors").is_file():
    root, subfolder = (checkpoint.parent, checkpoint.name) if (checkpoint.parent / "voices_v3_turbo.json").is_file() else (checkpoint, "")
else:
    raise ValueError("Cần checkpoint đã merge: model.safetensors, config.json, tokenizer.json; không phải adapter LoRA.")
for filename in ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"):
    if not (root / subfolder / filename).is_file():
        raise ValueError(f"Thiếu file: {root / subfolder / filename}")
voices_file = root / "voices_v3_turbo.json"
if not voices_file.is_file():
    raise ValueError(f"Thiếu {voices_file}. Upload file giọng đi kèm checkpoint vào thư mục merged.")
voices_document = json.loads(voices_file.read_text(encoding="utf-8"))
if not voices_document.get("presets") or voices_document.get("default_voice") not in voices_document["presets"]:
    raise ValueError("voices_v3_turbo.json cần presets và default_voice hợp lệ.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Chọn Runtime → Change runtime type → T4 GPU (hoặc GPU khác), rồi chạy lại.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
try:
    API_KEY = userdata.get(API_KEY_SECRET).strip() if API_KEY_SECRET else ""
except userdata.SecretNotFoundError:
    API_KEY = ""
except userdata.NotebookAccessError:
    API_KEY = ""
if not API_KEY:
    API_KEY = getpass.getpass("Nhập API key cho server (được ẩn): ").strip()
if not API_KEY or API_KEY == "replace_with_your_key":
    raise ValueError("Cần API key không rỗng để xuất server ra Internet.")

work = Path("/content/vieneu-colab-api")
work.mkdir(exist_ok=True)
logs = work / "logs"
logs.mkdir(exist_ok=True)
print("2/6 Chuẩn bị source...")
if SOURCE_DIR.strip():
    repo = Path(SOURCE_DIR.strip()).expanduser().resolve()
else:
    # Clone mới để không dùng nhầm branch của lần chạy trước; giữ các bản cũ.
    import tempfile
    repo = Path(tempfile.mkdtemp(prefix="source-", dir=work))
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF.strip(), "--", REPO_URL.strip(), str(repo)], check=True)
for filename in ("apps/model_profiles.py", "apps/openai_speech.py", "uv.lock", "pyproject.toml"):
    if not (repo / filename).is_file():
        raise ValueError(f"Source thiếu {filename}. Dùng repo/branch có tính năng local checkpoint (commit e8960c4 trở lên), hoặc upload source lên Drive và điền SOURCE_DIR.")

print("3/6 Cài môi trường Python 3.12 riêng và FFmpeg (lần đầu có thể mất vài phút)...")
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    raise RuntimeError("Không tìm thấy uv sau khi cài đặt.")
env = os.environ.copy()
env.update({"UV_PROJECT_ENVIRONMENT": str(work / "venv"), "UV_LINK_MODE": "copy"})
# Không thay torch của kernel Colab: server dùng Python/torch trong venv riêng.
subprocess.run([uv, "sync", "--project", str(repo), "--python", "3.12", "--frozen", "--no-dev", "--extra", "cuda", "--no-install-project"], env=env, check=True)
python = work / "venv/bin/python"
subprocess.run([str(python), "-c", "import torch; assert torch.cuda.is_available(), 'CUDA không khả dụng trong môi trường server'; print('CUDA:', torch.cuda.get_device_name(0))"], env=env, check=True)
# YAML viết dưới dạng JSON: JSON cũng là YAML hợp lệ, không cần import PyYAML vào kernel.
config = work / "local-profile.yaml"
config.write_text(json.dumps({"backbone_configs": {"colab-drive": {
    "mode": "v3turbo", "repo": str(root), "local": True,
    "backend": "pytorch", "requires_cuda": True,
    "model_subfolder": subfolder, "supports_streaming": False,
}}}, ensure_ascii=False), encoding="utf-8")
# Loại bỏ biến inference cũ nếu cell chạy lại trong kernel đã cấu hình trước đó.
for name in list(env):
    if name.startswith("VIENEU_"):
        del env[name]
env.update({
    "PYTHONPATH": os.pathsep.join([str(repo / "src"), str(repo)]),
    "PYTHONUNBUFFERED": "1", "HOST": "127.0.0.1", "PORT": "8000",
    "VIENEU_HOME": str(work / "data"), "VIENEU_VOICE_PRO_ENABLED": "1",
    "VIENEU_API_KEY": API_KEY, "VIENEU_BACKBONE": "v3turbo",
    "VIENEU_BACKEND": "pytorch", "VIENEU_DEVICE": "cuda",
    "VIENEU_CONFIG_PATH": str(config), "VIENEU_MODEL_PROFILE": "colab-drive",
    "VIENEU_TTS_TIMEOUT": "90", "VIENEU_MAX_AUDIO_SECONDS": "90",
})
LOCAL_URL = "http://127.0.0.1:8000"

def get_json(url, authenticated=False):
    headers = {"Authorization": "Bearer " + API_KEY} if authenticated else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=10) as response:
        return json.load(response)

def tail_log(path):
    return path.read_text(encoding="utf-8", errors="replace")[-6000:]

try:
    print("4/6 Load checkpoint và chờ server sẵn sàng...")
    server_log = logs / "server.log"
    server_log_handle = server_log.open("w", encoding="utf-8")
    server_process = subprocess.Popen([str(python), "-m", "apps.openai_speech"], cwd=repo, env=env, stdout=server_log_handle, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + max(60, STARTUP_TIMEOUT_SECONDS)
    next_progress = 0
    while True:
        if server_process.poll() is not None:
            raise RuntimeError("Server thoát khi startup:\n" + tail_log(server_log))
        try:
            get_json(LOCAL_URL + "/health")
            CAPABILITIES = get_json(LOCAL_URL + "/v1/tts/capabilities", True)
            break
        except (urllib.error.URLError, TimeoutError, OSError):
            if time.monotonic() >= deadline:
                raise RuntimeError("Hết thời gian load model:\n" + tail_log(server_log))
            if time.monotonic() >= next_progress:
                print("Đang load model / tải codec và speaker encoder lần đầu...")
                next_progress = time.monotonic() + 30
            time.sleep(2)
    if CAPABILITIES.get("model_profile") != "colab-drive" or Path(CAPABILITIES["backbone_repo"]).resolve() != root:
        raise RuntimeError("API không load đúng checkpoint đã chọn.")

    print("5/6 Mở Cloudflare Quick Tunnel...")
    cloudflared = work / "cloudflared"
    if not cloudflared.is_file():
        download = work / "cloudflared.download"
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", download)
        download.chmod(0o755)
        download.replace(cloudflared)
    tunnel_log = logs / "tunnel.log"
    tunnel_log_handle = tunnel_log.open("w", encoding="utf-8")
    tunnel_process = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", LOCAL_URL], stdout=tunnel_log_handle, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 120
    last_error = ""
    while time.monotonic() < deadline:
        if server_process.poll() is not None or tunnel_process.poll() is not None:
            raise RuntimeError("Server hoặc tunnel đã dừng:\n" + tail_log(tunnel_log) + "\n" + tail_log(server_log))
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail_log(tunnel_log))
        if match:
            candidate = match.group(0)
            try:
                public_capabilities = get_json(candidate + "/v1/tts/capabilities", True)
                if public_capabilities["model"] == CAPABILITIES["model"]:
                    PUBLIC_URL = candidate
                    break
            except (urllib.error.URLError, TimeoutError, OSError, ValueError) as exc:
                last_error = str(exc)
        time.sleep(3)
    if not PUBLIC_URL:
        raise RuntimeError("Tunnel chưa truy cập được: " + last_error + "\n" + tail_log(tunnel_log))
    BASE_URL = PUBLIC_URL + "/v1"
    print("6/6 SẴN SÀNG")
    print("API URL:", PUBLIC_URL)
    print("OpenAI Base URL:", BASE_URL)
    print("TTS endpoint:", PUBLIC_URL + "/v1/tts")
    print("Model:", CAPABILITIES["model"])
    print("Giọng:", ", ".join(voices_document["presets"]))
    print("API key: dùng giá trị bạn vừa nhập / Colab Secret (không in ra output).")
    print("Server chạy nền trong phiên Colab. Dừng bằng cell Dừng bên dưới.")
    print("Log:", server_log, tunnel_log)
except BaseException:
    stop_server()
    PUBLIC_URL = BASE_URL = None
    raise


In [ ]:
# @title Lấy danh sách voices từ API
if not globals().get("PUBLIC_URL"):
    raise RuntimeError("Chạy cell khởi động API trước.")
VOICES = get_json(PUBLIC_URL + "/v1/voices", authenticated=True)["data"]
print("Các giọng server đang cung cấp (dùng ID khi gọi API):")
for voice in VOICES:
    print("ID:", voice["id"], "| Tên:", voice.get("name", voice["id"]))
    if voice.get("description"):
        print("  ", voice["description"])
VOICE_IDS = [voice["id"] for voice in VOICES]
if not VOICE_IDS:
    print("Server chưa có giọng nào.")


In [ ]:
# @title Tạo audio mẫu (tùy chọn)
TEXT = "Xin chào, đây là API VieNeu chạy trên Google Colab." # @param {type:"string"}
VOICE = "" # @param {type:"string"}
TEMPERATURE = 0.8 # @param {type:"number"}
from IPython.display import Audio, display
if not PUBLIC_URL:
    raise RuntimeError("Chạy cell khởi động trước.")
request = urllib.request.Request(
    PUBLIC_URL + "/v1/tts",
    data=json.dumps({"text": TEXT, "voice": VOICE or voices_document["default_voice"], "format": "wav", "temperature": TEMPERATURE}).encode("utf-8"),
    headers={"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=100) as response:
    audio = response.read()
output = Path("/content/vieneu-sample.wav")
output.write_bytes(audio)
display(Audio(audio))
print("Đã lưu:", output)


In [ ]:
# @title Dừng API và Cloudflare Tunnel
if "stop_server" in globals():
    stop_server()
PUBLIC_URL = BASE_URL = None
print("Đã dừng server và tunnel của notebook.")
